# Análisis del benchmark ICDAS v4 — figuras y Grad-CAM
**Settings:** GPU T4 (una basta), Internet ON (instala timm). **Inputs:** salida de `benchmark-icdas-v4` (predicciones fuera de fold, métricas, pesos) y salida de `base-icdas-v4` (recortes).

Contenido:
1. Regla fijada para elegir el clasificador del pipeline: mayor macro-F1 de **validación** (media de los 5 folds).
2. Figuras: forest plot de macro-F1, matrices de confusión, ICDAS 0–6 × predicción, curvas ROC en los tres cortes, diagrama de calibración, eficiencia vs desempeño.
3. **Grad-CAM** con el modelo del fold en que cada recorte fue prueba (sin fuga): mismos recortes para los 6 modelos de timm/torchvision (aciertos por clase + errores típicos) y una medida simple de dónde mira cada modelo (fracción del mapa en el centro del recorte, 300 recortes al azar por modelo). YOLO11m-cls queda fuera del Grad-CAM.

In [ ]:
!pip -q install timm albumentations
import os, glob, json, re, zipfile, math, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm, cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchvision.models as tvm
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
IMG, CLASSES = 224, ['0', 'A', 'B', 'C']
FIG = f'{W}/figuras'; os.makedirs(FIG, exist_ok=True)
find = lambda n: (glob.glob(f'{IN}/**/{n}', recursive=True) or [None])[0]
RES = os.path.dirname(find('oof_predicciones.csv'))
NAME = {'swin_tiny': 'Swin-T', 'convnextv2_tiny': 'ConvNeXt V2-T', 'maxvit_tiny': 'MaxViT-T', 'coatnet_0': 'CoAtNet-0',
        'yolo11m_cls': 'YOLO11m-cls', 'vit_small': 'ViT-S/16', 'resnet50': 'ResNet50'}
oof = pd.read_csv(f'{RES}/oof_predicciones.csv'); res = pd.read_csv(f'{RES}/metricas_oof_ic95.csv'); cmp = pd.read_csv(f'{RES}/comparacion_pareada.csv')
ORDER = res.sort_values('macro_f1', ascending=False).model.tolist()
print(oof.groupby('model').size().to_dict()); print(ORDER)
def save(fig, name):
    fig.savefig(f'{FIG}/{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{FIG}/{name}.pdf', bbox_inches='tight'); plt.close(fig)

## 1. Regla fijada: mejor macro-F1 de validación

In [ ]:
meta = pd.DataFrame([json.load(open(f)) for f in glob.glob(f'{RES}/*/fold*_meta.json')])
val = meta.groupby('model').agg(val_macro_f1=('best_val_macro_f1', 'mean'), val_sd=('best_val_macro_f1', 'std'), mejor_epoca=('best_epoch', 'median'),
                                min_entrenamiento=('train_time_s', lambda s: s.sum() / 60)).sort_values('val_macro_f1', ascending=False)
val = val.join(res.set_index('model')[['macro_f1']].rename(columns={'macro_f1': 'test_oof_macro_f1'}))
val.to_csv(f'{W}/seleccion_por_validacion.csv'); print(val.round(3).to_string())
print('Elegido para el pipeline (regla de validación):', val.index[0])

## 2. Figuras

In [ ]:
# 2a. Forest plot de macro-F1 (IC 95 % bootstrap por niño), marca de diferencia significativa vs el mejor (Holm)
sig = set(cmp[cmp.p_holm < 0.05].vs)
fig, ax = plt.subplots(figsize=(6, 3.2))
for i, m in enumerate(ORDER[::-1]):
    r = res.set_index('model').loc[m]
    ax.errorbar(r.macro_f1, i, xerr=[[r.macro_f1 - r.macro_f1_lo], [r.macro_f1_hi - r.macro_f1]], fmt='o', color='#444' if m in sig else '#1d5d74', capsize=3)
    ax.text(r.macro_f1_hi + 0.002, i, f'{r.macro_f1:.3f}' + (' *' if m in sig else ''), va='center', fontsize=8)
ax.set_yticks(range(len(ORDER))); ax.set_yticklabels([NAME[m] for m in ORDER[::-1]]); ax.set_xlabel('Macro-F1 (Merged-ICDAS), out-of-fold, 95% CI')
ax.grid(axis='x', alpha=.3); save(fig, 'F1_forest_macroF1')

# 2b. Matrices de confusión normalizadas por fila (Merged-ICDAS)
fig, axs = plt.subplots(2, 4, figsize=(13, 6.4))
for ax, m in zip(axs.flat, ORDER):
    d = oof[oof.model == m]; cm = confusion_matrix(d.y, d.pred, labels=[0, 1, 2, 3]); cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(4):
        for j in range(4): ax.text(j, i, f'{cmn[i, j]:.2f}\n({cm[i, j]})', ha='center', va='center', fontsize=7, color='white' if cmn[i, j] > .5 else 'black')
    ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES); ax.set_title(NAME[m], fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Reference')
axs.flat[-1].axis('off'); fig.tight_layout(); save(fig, 'F2_confusion_merged')

# 2c. ICDAS 0-6 de referencia x clase predicha (mejor modelo)
best = ORDER[0]; d = oof[oof.model == best]
ct = pd.crosstab(d.icdas, d.pred.map(dict(enumerate(CLASSES)))).reindex(columns=CLASSES, fill_value=0); ctn = ct.div(ct.sum(1), axis=0)
fig, ax = plt.subplots(figsize=(4.2, 4.2)); ax.imshow(ctn.values, cmap='Blues', vmin=0, vmax=1)
for i in range(ct.shape[0]):
    for j in range(4): ax.text(j, i, f'{ctn.values[i, j]:.2f}\n({ct.values[i, j]})', ha='center', va='center', fontsize=7, color='white' if ctn.values[i, j] > .5 else 'black')
ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(ct.shape[0])); ax.set_yticklabels(ct.index)
ax.set_xlabel('Predicted Merged-ICDAS'); ax.set_ylabel('Reference ICDAS code'); ax.set_title(NAME[best], fontsize=9); save(fig, 'F3_icdas_codes_vs_pred')

# 2d. ROC en los tres cortes
CUTS = [('ICDAS ≥ 1', 1), ('ICDAS ≥ 3', 2), ('ICDAS ≥ 5', 3)]
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, (nm, c0) in zip(axs, CUTS):
    for m in ORDER:
        d = oof[oof.model == m]; t = d.y >= c0; s = d[[f'p_{c}' for c in CLASSES[c0:]]].sum(1)
        fpr, tpr, _ = roc_curve(t, s); ax.plot(fpr, tpr, lw=1.2, label=f'{NAME[m]} ({roc_auc_score(t, s):.3f})')
        sp = ((~t) & (d.pred < c0)).sum() / (~t).sum(); se = (t & (d.pred >= c0)).sum() / t.sum(); ax.plot(1 - sp, se, 'o', ms=3, color=ax.lines[-1].get_color())
    ax.plot([0, 1], [0, 1], ':', color='gray'); ax.set_title(nm); ax.set_xlabel('1 − specificity'); ax.set_ylabel('Sensitivity'); ax.legend(fontsize=6.5, loc='lower right')
fig.tight_layout(); save(fig, 'F4_roc_cuts')

# 2e. Diagrama de calibración (confianza de la clase predicha)
fig, ax = plt.subplots(figsize=(4.5, 4.5)); bins = np.linspace(0, 1, 11)
for m in ORDER:
    d = oof[oof.model == m]; P = d[[f'p_{c}' for c in CLASSES]].values; conf = P.max(1); ok = P.argmax(1) == d.y.values
    idx = np.digitize(conf, bins) - 1; xs, ys = [], []
    for b in range(10):
        k = idx == b
        if k.sum() >= 10: xs.append(conf[k].mean()); ys.append(ok[k].mean())
    ax.plot(xs, ys, 'o-', ms=3, lw=1, label=f"{NAME[m]} (ECE {res.set_index('model').ece[m]:.3f})")
ax.plot([0, 1], [0, 1], ':', color='gray'); ax.set_xlabel('Mean predicted confidence'); ax.set_ylabel('Observed accuracy'); ax.legend(fontsize=6.5); save(fig, 'F5_calibration')

# 2f. Eficiencia vs desempeño
eff = pd.DataFrame([json.load(open(f)) for f in glob.glob(f'{RES}/*/efficiency.json')]).set_index('model')
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, lab in ((axs[0], 'gflops', 'GFLOPs per image'), (axs[1], 'cpu_ms_per_img_1thread', 'CPU latency (ms/image, 1 thread)')):
    for m in ORDER:
        ax.scatter(eff.loc[m, col], res.set_index('model').macro_f1[m], s=30); ax.annotate(NAME[m], (eff.loc[m, col], res.set_index('model').macro_f1[m]), fontsize=7, xytext=(3, 3), textcoords='offset points')
    ax.set_xlabel(lab); ax.set_ylabel('Macro-F1'); ax.grid(alpha=.3)
fig.tight_layout(); save(fig, 'F6_efficiency'); eff.to_csv(f'{W}/eficiencia.csv'); print(eff.round(2))

## 3. Grad-CAM (modelo del fold en que el recorte fue prueba)

In [ ]:
HEAVY_BOOST  = {0: 1.5, 1: 1.0, 2: 0.7, 3: 1.8}   # ResNet50 v6 / ConvNeXt V2 v11 / ViT-S v1b / Swin-T v1 — en el sampler
HYBRID_BOOST = {0: 1.0, 1: 1.3, 2: 1.0, 3: 2.0}   # CoAtNet-0 v2 / MaxViT-T — como peso de la loss
RECIPES = {
 'resnet50': dict(src='tv', ckpt='resnet50.IMAGENET1K_V2', head='mlp_bn', lr=3e-4, wd=3e-4, bs=32, epochs=50, patience=15,
                  warmup=5, min_lr=1e-6, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2, clip=2.0,
                  boost=HEAVY_BOOST, boost_in='sampler', aug='heavy_ge'),
 'convnextv2_tiny': dict(src='timm', ckpt='convnextv2_tiny.fcmae_ft_in1k', head='ln_drop', drop=0.4, lr=1e-4, wd=5e-4, bs=16,
                  epochs=60, patience=15, warmup=8, min_lr=1e-7, warm_style='e', freeze=8, unfreeze_stages=3, unfreeze='resume',
                  loss='focal', gamma=2.0, ls=0.0, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.2, clip=None,
                  boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'vit_small': dict(src='timm', ckpt='vit_small_patch16_224.augreg_in1k', head='linear', lr=1e-4, wd=0.05, bs=32, epochs=50,
                  patience=15, warmup=5, min_lr=1e-7, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2,
                  clip=1.0, boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'swin_tiny': dict(src='timm', ckpt='swin_tiny_patch4_window7_224.ms_in1k', head='linear', lr=1e-4, wd=0.05, bs=32, epochs=50,
                  patience=15, warmup=5, min_lr=1e-7, warm_style='e', freeze=0, loss='ce', ls=0.1, mix='mixup', mixup_a=0.2,
                  clip=1.0, boost=HEAVY_BOOST, boost_in='sampler', aug='heavy'),
 'coatnet_0': dict(src='timm', ckpt='coatnet_0_rw_224.sw_in1k', head='ln_mlp', lr=3e-4, wd=0.05, bs=32, epochs=45, patience=12,
                  warmup=5, min_lr=0.0, warm_style='e+1', freeze=6, unfreeze_stages=2, unfreeze='restart_half',
                  loss='focal', gamma=2.0, ls=0.05, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.3, clip=1.0,
                  boost=HYBRID_BOOST, boost_in='loss', aug='hybrid', drop_last=True),
 'maxvit_tiny': dict(src='timm', ckpt='maxvit_tiny_tf_224.in1k', head='ln_mlp', lr=3e-4, wd=0.05, bs=32, epochs=45, patience=12,
                  warmup=5, min_lr=0.0, warm_style='e+1', freeze=6, unfreeze_stages=2, unfreeze='restart_half',
                  loss='focal', gamma=2.0, ls=0.05, mix='cutmix_mixup', cutmix_p=0.5, mixup_a=0.3, clip=1.0,
                  boost=HYBRID_BOOST, boost_in='loss', aug='hybrid', drop_last=True),
 'yolo11m_cls': dict(src='ultralytics', ckpt='yolo11m-cls.pt', epochs=100, imgsz=224, batch=16, recipe='Ultralytics defaults'),
}


class Net(nn.Module):
    def __init__(self, r):
        super().__init__()
        if r['src'] == 'tv':
            self.backbone = tvm.resnet50(weights=None); nf = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        else:
            self.backbone = timm.create_model(r['ckpt'], pretrained=False, num_classes=0); nf = self.backbone.num_features
        h = r['head']
        if h == 'mlp_bn':  self.head = nn.Sequential(nn.Linear(nf, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 4))
        elif h == 'ln_drop': self.head = nn.Sequential(nn.LayerNorm(nf, eps=1e-6), nn.Dropout(r['drop']), nn.Linear(nf, 4))
        elif h == 'ln_mlp':  self.head = nn.Sequential(nn.LayerNorm(nf), nn.Dropout(0.5), nn.Linear(nf, 512), nn.GELU(), nn.Dropout(0.2), nn.Linear(512, 4))
        else:                self.head = nn.Linear(nf, 4)
    def forward(self, x): return self.head(self.backbone(x))
    def freeze(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_last(self, n):
        for st in list(self.backbone.stages)[-n:]:
            for p in st.parameters(): p.requires_grad = True
        for nm in ('norm_pre', 'norm'):
            if hasattr(self.backbone, nm) and isinstance(getattr(self.backbone, nm), nn.Module):
                for p in getattr(self.backbone, nm).parameters(): p.requires_grad = True


DATA = f'{TMP}/icdas'
if not os.path.exists(f'{DATA}/icdas_v4'): zipfile.ZipFile(find('icdas_v4.zip')).extractall(DATA)
man = pd.read_csv(f'{DATA}/icdas_v4/manifest.csv', dtype={'micdas': str}).set_index('crop_id')
def load_img(cid): return cv2.cvtColor(cv2.imread(f'{DATA}/icdas_v4/' + man.loc[cid, 'crop_file']), cv2.COLOR_BGR2RGB)

def target_layer(m, net):
    b = net.backbone
    if m == 'resnet50': return b.layer4
    if m == 'swin_tiny': return b.layers[-1]
    if m == 'vit_small': return b.blocks[-1].norm1
    return b.stages[-1]
def to_nchw(m, a):
    if m == 'swin_tiny': return a.permute(0, 3, 1, 2)
    if m == 'vit_small':
        a = a[:, getattr(models_cache.get(m, (None,))[0].backbone, 'num_prefix_tokens', 1):]; s = int(math.sqrt(a.shape[1])); return a.reshape(a.shape[0], s, s, -1).permute(0, 3, 1, 2)
    return a

models_cache = {}
def get_model(m, k):
    key = (m, k)
    if key in models_cache: return models_cache[key]
    meta_k = json.load(open(f'{RES}/{m}/fold{k}_meta.json')); net = Net(RECIPES[m])
    sd = torch.load(f'{RES}/{m}/fold{k}_weights_fp16.pt', map_location='cpu'); net.load_state_dict({a: v.float() for a, v in sd.items()})
    net = net.to(DEV).eval(); tf = A.Compose([A.Resize(IMG, IMG), A.Normalize(mean=meta_k['mean'], std=meta_k['std']), ToTensorV2()])
    models_cache.clear(); models_cache[key] = (net, tf); models_cache[m] = (net, tf); return net, tf

def gradcam(m, k, cid, cls=None):
    net, tf = get_model(m, k); acts, grads = {}, {}
    L = target_layer(m, net)
    h1 = L.register_forward_hook(lambda mod, i, o: acts.__setitem__('a', o))
    h2 = L.register_full_backward_hook(lambda mod, gi, go: grads.__setitem__('g', go[0]))
    x = tf(image=load_img(cid))['image'][None].to(DEV).requires_grad_(True)
    out = net(x); p = torch.softmax(out, 1)[0].detach().cpu().numpy(); c = int(out.argmax(1)) if cls is None else cls
    net.zero_grad(); out[0, c].backward(); h1.remove(); h2.remove()
    a, g = to_nchw(m, acts['a']).detach(), to_nchw(m, grads['g']).detach()
    cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1))[0].cpu().numpy()
    cam = cv2.resize(cam, (IMG, IMG)); cam = cam / cam.max() if cam.max() > 0 else cam
    return cam, p, c

def overlay(cid, cam):
    im = cv2.resize(load_img(cid), (IMG, IMG)); hm = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[:, :, ::-1]
    return (0.55 * im + 0.45 * hm).astype(np.uint8)

In [ ]:
CAM_MODELS = [m for m in ORDER if m != 'yolo11m_cls']
P = oof.pivot_table(index='crop_id', columns='model', values='pred'); Y = oof.groupby('crop_id').y.first(); FOLD = oof.groupby('crop_id').fold.first()
ICD = oof.groupby('crop_id').icdas.first(); conf_best = oof[oof.model == best].set_index('crop_id')[[f'p_{c}' for c in CLASSES]].max(1)
rng = np.random.default_rng(2026); sel = []
for i, c in enumerate(CLASSES):   # aciertos claros: todos los modelos aciertan, alta confianza del mejor
    ok = P.index[(P[CAM_MODELS] == i).all(1) & (Y == i)]; ok = conf_best.loc[ok].sort_values(ascending=False).index[:20]
    sel += [(cid, f'{c} correct') for cid in rng.choice(ok, min(2, len(ok)), replace=False)]
for t, p_, lab in ((2, 1, 'B→A'), (1, 0, 'A→0'), (3, 2, 'C→B'), (0, 1, '0→A'), (1, 2, 'A→B')):   # errores típicos del mejor modelo
    e = P.index[(Y == t) & (P[best] == p_)]
    sel += [(cid, f'{lab} error (ICDAS {ICD[cid]})') for cid in rng.choice(e, min(2, len(e)), replace=False)]
print(len(sel), 'recortes para Grad-CAM'); pd.DataFrame(sel, columns=['crop_id', 'grupo']).to_csv(f'{W}/gradcam_seleccion.csv', index=False)
R = {}
for m in CAM_MODELS:
    for cid, g in sorted(sel, key=lambda s: FOLD[s[0]]): R[(m, cid)] = gradcam(m, FOLD[cid], cid)
n = len(sel); fig, axs = plt.subplots(n, len(CAM_MODELS) + 1, figsize=(1.9 * (len(CAM_MODELS) + 1), 1.95 * n), squeeze=False)
for r, (cid, g) in enumerate(sel):
    axs[r, 0].imshow(cv2.resize(load_img(cid), (IMG, IMG))); axs[r, 0].set_ylabel(g, fontsize=7)
    for j, m in enumerate(CAM_MODELS):
        cam, p, c = R[(m, cid)]; axs[r, j + 1].imshow(overlay(cid, cam)); axs[r, j + 1].set_xlabel(f'{CLASSES[c]} {p[c]:.2f}', fontsize=7)
        if r == 0: axs[r, j + 1].set_title(NAME[m], fontsize=8)
    axs[0, 0].set_title('Crop', fontsize=8)
for ax in axs.flat: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); save(fig, 'F7_gradcam_panel')
# panel corto para el paper: aciertos (1 por clase) + 4 errores, mejor modelo y uno por familia
fam = [m for m in (best, 'convnextv2_tiny', 'maxvit_tiny', 'resnet50') if m in CAM_MODELS]; fam = list(dict.fromkeys(fam))
short = [s for s in sel if 'correct' in s[1]][::2] + [s for s in sel if 'error' in s[1]][::2][:4]
fig, axs = plt.subplots(len(short), len(fam) + 1, figsize=(1.9 * (len(fam) + 1), 1.95 * len(short)), squeeze=False)
for r, (cid, g) in enumerate(short):
    axs[r, 0].imshow(cv2.resize(load_img(cid), (IMG, IMG))); axs[r, 0].set_ylabel(g, fontsize=7)
    for j, m in enumerate(fam):
        cam, p, c = R[(m, cid)]; axs[r, j + 1].imshow(overlay(cid, cam)); axs[r, j + 1].set_xlabel(f'{CLASSES[c]} {p[c]:.2f}', fontsize=7)
        if r == 0: axs[r, j + 1].set_title(NAME[m], fontsize=8)
    axs[0, 0].set_title('Crop', fontsize=8)
for ax in axs.flat: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); save(fig, 'F8_gradcam_paper')

### ¿Dónde mira cada modelo? Fracción del mapa Grad-CAM en el centro del recorte
Los recortes están centrados en el diente; los bordes suelen tener encía, retractor o dientes vecinos. Se mide, en 300 recortes al azar (estratificados por clase), la fracción de la energía del mapa dentro del 60 % central del recorte. Es una medida descriptiva, no una prueba de que el modelo mire la lesión.

In [ ]:
samp = pd.concat([g.sample(75, random_state=1) for _, g in oof[oof.model == best].groupby('y')]).crop_id.tolist()
mask = np.zeros((IMG, IMG)); a = int(IMG * .2); mask[a:IMG - a, a:IMG - a] = 1
rows = []
for m in CAM_MODELS:
    for cid in sorted(samp, key=lambda c: FOLD[c]):
        cam, p, c = gradcam(m, FOLD[cid], cid); rows.append(dict(model=m, crop_id=cid, y=Y[cid], acierto=c == Y[cid], centro=(cam * mask).sum() / max(1e-9, cam.sum())))
cen = pd.DataFrame(rows); cen.to_csv(f'{W}/gradcam_centralidad.csv', index=False)
print('Fracción del mapa en el 60 % central (área = 36 %):'); print(cen.groupby('model').centro.describe()[['mean', '25%', '50%', '75%']].round(3).loc[CAM_MODELS])
print(cen.groupby(['model', 'acierto']).centro.mean().unstack().round(3))
print({f: round(os.path.getsize(f'{FIG}/{f}') / 1e6, 2) for f in sorted(os.listdir(FIG))})